# DAIS 2026 Datathon — Virtue Foundation Dataset Exploration

Datasource: shared Delta Sharing catalog `databricks_virtue_foundation_dataset_dais_2026`, schema `virtue_foundation_dataset`, three tables:

* `facilities` — health-facility directory (name, organization type, address, contact info, coordinates)
* `india_post_pincode_directory` — India Post pincode → post office / district / state lookup
* `nfhs_5_district_health_indicators` — NFHS-5 district-level health, maternity and nutrition indicators (~109 columns)

Note: in `nfhs_5_district_health_indicators` many `_pct` columns are typed `string` (some values carry confidence-interval annotations like `42.5 (40.1, 44.9)`), so we cast them defensively with a regex that extracts the leading number.

## 1. List the tables and their schemas

In [0]:
CATALOG = "databricks_virtue_foundation_dataset_dais_2026"
SCHEMA = "virtue_foundation_dataset"

from pyspark.sql import functions as F

display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}"))

for t in ["facilities", "india_post_pincode_directory", "nfhs_5_district_health_indicators"]:
    print(f"\n{'=' * 90}\nSchema of {CATALOG}.{SCHEMA}.{t}\n{'=' * 90}")
    spark.table(f"{CATALOG}.{SCHEMA}.{t}").printSchema()

## 2. Load the tables — row counts and samples

In [0]:
facilities = spark.table(f"{CATALOG}.{SCHEMA}.facilities")
pincode = spark.table(f"{CATALOG}.{SCHEMA}.india_post_pincode_directory")
nfhs = spark.table(f"{CATALOG}.{SCHEMA}.nfhs_5_district_health_indicators")

for name, df in [("facilities", facilities),
                 ("india_post_pincode_directory", pincode),
                 ("nfhs_5_district_health_indicators", nfhs)]:
    print(f"{name:40s} rows: {df.count():>10,}   columns: {len(df.columns)}")

print("\nfacilities sample:")
display(facilities.limit(10))

print("\nindia_post_pincode_directory sample:")
display(pincode.limit(10))

print("\nnfhs_5_district_health_indicators sample (selected columns):")
display(nfhs.limit(10))

## 3. NFHS-5 EDA — district/state coverage and missing values

In [0]:
print(f"Rows: {nfhs.count():,}")
print(f"Distinct state/UT values: {nfhs.select('state_ut').distinct().count()}")
print(f"Distinct district names: {nfhs.select('district_name').distinct().count()}")
print(f"Distinct (state, district) pairs: {nfhs.select('state_ut', 'district_name').distinct().count()}")

n_dupes = nfhs.groupBy("state_ut", "district_name").count().where(F.col("count") > 1).count()
print(f"Duplicate (state, district) rows: {n_dupes}")

coverage = (nfhs.groupBy("state_ut")
            .agg(F.count("*").alias("n_district_rows"),
                 F.round(F.sum("households_surveyed")).alias("households_surveyed"),
                 F.round(F.avg("institutional_birth_5y_pct"), 1).alias("avg_institutional_birth_pct"))
            .orderBy(F.desc("n_district_rows")))
display(coverage)

In [0]:
# Percent of NULLs per column: one aggregated row, pivoted with pandas
null_row = nfhs.agg(*[F.round(F.avg(F.col(c).isNull().cast("int")) * 100, 1).alias(c) for c in nfhs.columns])
missing = (null_row.toPandas().T
           .rename(columns={0: "pct_missing"})
           .sort_values("pct_missing", ascending=False))
missing.index.name = "column"
print(f"Columns with 0% missing: {(missing.pct_missing == 0).sum()} of {len(missing)}")
print("Top 15 columns by missingness:")
display(missing.head(15))

# Several pct columns are typed string — inspect raw values of a key one before casting
print("Distinct raw values of child_u5_who_are_stunted_height_for_age_18_pct (first 5):")
nfhs.select("child_u5_who_are_stunted_height_for_age_18_pct").distinct().show(5, truncate=False)

# Robust cast helper: extract the leading decimal number; CI annotations / '*' become NULL
def pct(colname):
    s = F.regexp_extract(F.col(colname), r"^\s*(-?\d+(?:\.\d+)?)", 1)
    return F.when(F.length(s) > 0, s.cast("double"))

print("Same columns after the defensive cast:")
nfhs.select("state_ut", "district_name",
            pct("child_u5_who_are_stunted_height_for_age_18_pct").alias("stunted_pct"),
            pct("institutional_birth_5y_pct").alias("inst_birth_pct")).show(5)

## 4. Key indicators — top/bottom states (unweighted district means)

In [0]:
KEY_INDICATORS = {
    "institutional_birth_5y_pct": "Institutional births in the last 5 years (%)",
    "child_u5_who_are_stunted_height_for_age_18_pct": "Under-5 children stunted (%)",
}

for colname, label in KEY_INDICATORS.items():
    print(f"\n=== {label} — mean across districts, unweighted ===")
    by_state = (nfhs.groupBy("state_ut")
                .agg(F.round(F.avg(pct(colname)), 1).alias("state_avg_pct"),
                     F.count(pct(colname)).alias("districts_with_value")))
    top5 = by_state.orderBy(F.desc("state_avg_pct")).limit(5).withColumn("group", F.lit("top 5"))
    bottom5 = by_state.orderBy(F.asc("state_avg_pct")).limit(5).withColumn("group", F.lit("bottom 5"))
    display(top5.unionByName(bottom5))

## 5. Sketch — joining facilities ↔ pincode directory ↔ NFHS-5

Path: `facilities.address_zipOrPostcode` (string) → `india_post_pincode_directory.pincode`, aggregate facilities per district, then attempt a name-based match to NFHS-5 (`district_name`, `state_ut`). Expect mismatches from legacy India Post state/district naming and plain name differences — the match rates printed below quantify how rough this join is.

In [0]:
print("Facilities by address_country (top 10):")
facilities.groupBy("address_country").count().orderBy(F.desc("count")).show(10, truncate=False)

# India pincodes are exactly 6 digits
pin_str = F.regexp_extract("address_zipOrPostcode", r"^\s*(\d{6})$", 1)
fac_pin = (facilities
           .withColumn("pin", F.when(F.length(pin_str) > 0, pin_str.cast("long")))
           .where(F.col("pin").isNotNull()))
print(f"\nFacilities with a 6-digit pincode: {fac_pin.count():,} / {facilities.count():,}")

pin_dir = pincode.select("pincode", "officename", "district", "statename").distinct()

fac_geo = (fac_pin.alias("f")
           .join(pin_dir.alias("p"), F.col("f.pin") == F.col("p.pincode"), "left")
           .select(F.col("f.name").alias("facility_name"), "pin",
                   F.col("p.district").alias("post_district"),
                   F.col("p.statename").alias("post_state")))

n_total = fac_geo.count()
n_matched = fac_geo.where("post_district is not null").count()
print(f"Facilities matched to a pincode-directory district: {n_matched:,} / {n_total:,} ({100 * n_matched / n_total:.1f}%)")

# Aggregate facilities per district, then sketch the link to NFHS-5
fac_by_district = (fac_geo.where("post_district is not null")
                   .groupBy("post_state", "post_district")
                   .agg(F.count("*").alias("n_facilities")))

nfhs_keys = nfhs.select(
    F.lower(F.trim("district_name")).alias("d"),
    F.lower(F.trim("state_ut")).alias("s"),
    pct("institutional_birth_5y_pct").alias("institutional_birth_5y_pct"))

sketch = (fac_by_district
          .withColumn("d", F.lower(F.trim("post_district")))
          .withColumn("s", F.lower(F.trim("post_state")))
          .join(nfhs_keys, ["d", "s"], "left")
          .orderBy(F.desc("n_facilities")))

n_dist = sketch.count()
n_dist_matched = sketch.where("institutional_birth_5y_pct is not null").count()
print(f"\nDistricts with facilities that also match an NFHS-5 (state, district) pair: {n_dist_matched:,} / {n_dist:,} ({100 * n_dist_matched / n_dist:.1f}%)")
display(sketch.select("post_state", "post_district", "n_facilities",
                      "institutional_birth_5y_pct").limit(15))